# CXR Analyzer — Demo Gradio (Colab)
Xception M2 U-Ignore + Grad-CAM + MedSAM · Projeto Integrador SENAI FATESG 2025/2026

## SEÇÃO 0 — Instalações

In [ ]:
!pip install -q gradio tensorflow pillow matplotlib numpy
!pip install -q git+https://github.com/bowang-lab/MedSAM
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 128.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 121.7 MB/s eta 0:00:00


In [ ]:
import os, json, io, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import gradio as gr
from PIL import Image, ImageDraw, ImageFilter
import tensorflow as tf
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

SEED = 42
tf.random.set_seed(SEED)
print(f'TF {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

Mounted at /content/drive
TF 2.20.0 | GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## SEÇÃO 1 — Configuração de Caminhos

In [ ]:
# ── Edite os caminhos abaixo ──────────────────────────────────────────────────
BASE = '/content/drive/MyDrive'

M2_MODEL_PATH      = f'{BASE}/datasets_pi_2026/modelos/M2_NIH_CheXpert_Crop_UIgnore/m2_crop_final.keras'
MEDSAM_CKPT_PATH   = f'{BASE}/medsam_vit_b.pth'   # baixe se ainda não tiver

LABELS = ['atelectasis','cardiomegaly','pleural_effusion',
          'pneumothorax','consolidation','edema']
LABEL_PT = {
    'atelectasis':     'Atelectasia',
    'cardiomegaly':    'Cardiomegalia',
    'pleural_effusion':'Derrame Pleural',
    'pneumothorax':    'Pneumotórax',
    'consolidation':   'Consolidação',
    'edema':           'Edema',
}
LABEL_COLORS_HEX = {
    'atelectasis':     '#2E86AB',
    'cardiomegaly':    '#E76F51',
    'pleural_effusion':'#28A745',
    'pneumothorax':    '#FFC107',
    'consolidation':   '#6C63FF',
    'edema':           '#DC3545',
}
IMG_SIZE = 512
MEAN, STD = 0.5056, 0.2520

print('Verificando modelos:')
print(f'  M2    : {"✅" if os.path.exists(M2_MODEL_PATH) else "não encontrado"} ({M2_MODEL_PATH.split("/")[-1]})')
print(f'  MedSAM: {"✅" if os.path.exists(MEDSAM_CKPT_PATH) else "não encontrado (fallback visual)"}')

Verificando modelos:
  M2    : ✅ (m2_crop_final.keras)
  MedSAM: ✅


## SEÇÃO 2 — Classes Customizadas do M2

In [ ]:
N = len(LABELS)

class FocalLossUIgnore(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, label_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.gamma=gamma; self.alpha=alpha
        self.label_weights=(tf.constant(label_weights,dtype=tf.float32)
                            if label_weights is not None else tf.ones(N))
    def call(self,y_true,y_pred):
        labels=y_true[:,:N]; mask=y_true[:,N:]
        y_pred=tf.clip_by_value(y_pred,1e-7,1-1e-7)
        bce=-(labels*tf.math.log(y_pred)+(1-labels)*tf.math.log(1-y_pred))
        p_t=labels*y_pred+(1-labels)*(1-y_pred)
        a_t=labels*self.alpha+(1-labels)*(1-self.alpha)
        focal=a_t*tf.pow(1-p_t,self.gamma)*bce*self.label_weights
        return tf.reduce_sum(focal*mask)/(tf.reduce_sum(mask)+1e-7)
    def get_config(self):
        c=super().get_config()
        c.update({'gamma':self.gamma,'alpha':self.alpha,
                  'label_weights':self.label_weights.numpy().tolist()}); return c

class MacroAUROC(tf.keras.metrics.Metric):
    def __init__(self,n_labels=N,name='macro_auroc',**kwargs):
        super().__init__(name=name,**kwargs); self.n_labels=n_labels
        self.aucs=[tf.keras.metrics.AUC(curve='ROC',name=f'auc_{i}') for i in range(n_labels)]
    def update_state(self,y_true,y_pred,sample_weight=None):
        for i,auc in enumerate(self.aucs): auc.update_state(y_true[:,i],y_pred[:,i])
    def result(self): return tf.reduce_mean([a.result() for a in self.aucs])
    def reset_state(self):
        for a in self.aucs: a.reset_state()
    def get_config(self):
        c=super().get_config(); c['n_labels']=self.n_labels; return c

class MaskedBinaryAccuracy(tf.keras.metrics.Metric):
    def __init__(self,n_labels=N,name='accuracy',**kwargs):
        super().__init__(name=name,**kwargs); self.n_labels=n_labels
        self.acc=tf.keras.metrics.BinaryAccuracy()
    def update_state(self,y_true,y_pred,sample_weight=None):
        self.acc.update_state(y_true[:,:self.n_labels],y_pred)
    def result(self): return self.acc.result()
    def reset_state(self): self.acc.reset_state()
    def get_config(self):
        c=super().get_config(); c['n_labels']=self.n_labels; return c

CUSTOM = {'FocalLossUIgnore':FocalLossUIgnore,
          'MacroAUROC':MacroAUROC,
          'MaskedBinaryAccuracy':MaskedBinaryAccuracy}
print('Classes customizadas definidas.')

Classes customizadas definidas.


## SEÇÃO 3 — Carrega M2 e MedSAM

In [ ]:
# ── M2 Classifier ─────────────────────────────────────────────────────────────
print('Carregando M2...')
model_m2 = tf.keras.models.load_model(M2_MODEL_PATH,
                                       custom_objects=CUSTOM, compile=False)

# Grad-CAM: encontra última camada conv
GRADCAM_LAYER = None
for name in ['block14_sepconv2_act','block14_sepconv2_bn',
             'block14_sepconv1_act','block13_sepconv2_act']:
    try: model_m2.get_layer(name); GRADCAM_LAYER=name; break
    except ValueError: continue

grad_model = tf.keras.Model(
    inputs=model_m2.input,
    outputs=[model_m2.get_layer(GRADCAM_LAYER).output, model_m2.output]
)
print(f'M2 carregado | Grad-CAM layer: {GRADCAM_LAYER}')

Carregando M2...
M2 carregado | Grad-CAM layer: block14_sepconv2_act


In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

# Mirror com SHA256 verificado: 34b34b78c1d18cb8c6bf84cf9c00e135d6d6c965699f3c0e31ef1bc9dcb5be74
path = hf_hub_download(
    repo_id   = "SansuiHan/medical_models",
    filename  = "medsam_vit_b.pth",
    repo_type = "model",
)

# Copia para o Drive
dest = "/content/drive/MyDrive/medsam_vit_b.pth"
shutil.copy(path, dest)
print(f"Tamanho: {os.path.getsize(dest)/1024**2:.0f} MB")

medsam_vit_b.pth:   0%|          | 0.00/375M [00:00<?, ?B/s]

Tamanho: 358 MB


In [ ]:
import os

MEDSAM_CKPT = f'{BASE}/medsam_vit_b.pth'

# 1. Checkpoint existe?
print(f'Checkpoint existe: {os.path.exists(MEDSAM_CKPT)}')
if os.path.exists(MEDSAM_CKPT):
    print(f'Tamanho: {os.path.getsize(MEDSAM_CKPT)/1024**2:.0f} MB')  # deve ser ~375 MB

# 2. segment_anything instalado?
try:
    from segment_anything import sam_model_registry, SamPredictor
    print('segment_anything: OK')
except ImportError as e:
    print(f'segment_anything NAO instalado: {e}')

# 3. torch disponível com GPU?
try:
    import torch
    print(f'torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
except ImportError as e:
    print(f'torch NAO disponivel: {e}')

# 4. Tenta carregar o modelo
try:
    sam = sam_model_registry['vit_b'](checkpoint=MEDSAM_CKPT)
    print('sam_model_registry: carregou OK')
except Exception as e:
    print(f'Erro ao carregar modelo: {e}')

Checkpoint existe: True
Tamanho: 358 MB
segment_anything: OK
torch: 2.11.0+cu128 | CUDA: True
sam_model_registry: carregou OK


In [ ]:
# ── Instala MedSAM (só precisa rodar uma vez por sessão) ─────────────────────
!pip install -q git+https://github.com/bowang-lab/MedSAM
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

# ── Caminho do checkpoint MedSAM ─────────────────────────────────────────────
# Baixe com: !gdown https://huggingface.co/wanglab/medsam/resolve/main/medsam_vit_b.pth -O {BASE}/medsam_vit_b.pth
MEDSAM_CKPT = f'{BASE}/medsam_vit_b.pth'

  Preparing metadata (setup.py) ... done


In [ ]:
# ── MedSAM ────────────────────────────────────────────────────────────────────
MEDSAM_READY = False
medsam_predictor = None

if os.path.exists(MEDSAM_CKPT_PATH):
    try:
        import torch
        from segment_anything import sam_model_registry, SamPredictor
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        sam = sam_model_registry['vit_b'](checkpoint=MEDSAM_CKPT_PATH)
        sam.to(device)
        medsam_predictor = SamPredictor(sam)
        MEDSAM_READY = True
        print(f'MedSAM carregado ({device})')
    except Exception as e:
        print(f'MedSAM não carregado: {e}')
else:
    print('MedSAM checkpoint não encontrado — usando fallback visual.')
    print(f'Para baixar: gdown https://huggingface.co/wanglab/medsam/resolve/main/medsam_vit_b.pth')

MedSAM carregado (cuda)


## SEÇÃO 4 — Funções de Inferência

In [ ]:
def preprocess(pil_img):
    img = np.array(pil_img.convert('L').resize((IMG_SIZE,IMG_SIZE)),dtype=np.float32)/255.0
    img = (img-MEAN)/STD
    img = np.stack([img]*3, axis=-1)
    return tf.constant(img, dtype=tf.float32)


def get_gradcam(img_tensor, class_idx):
    """Grad-CAM padrão."""
    with tf.GradientTape() as tape:
        batch = tf.expand_dims(img_tensor,0)
        conv_out,preds = grad_model(batch)
        loss = preds[:,class_idx]
    grads  = tape.gradient(loss,conv_out)
    pooled = tf.reduce_mean(grads,axis=(0,1,2))
    hm     = conv_out[0] @ pooled[...,tf.newaxis]
    hm     = tf.squeeze(hm)
    hm     = tf.maximum(hm,0)/(tf.math.reduce_max(hm)+1e-7)
    return hm.numpy(), float(preds[0,class_idx])


def get_gradcam_pp(img_tensor, class_idx):
    """
    Grad-CAM++ (Chattopadhay et al., 2018).
    Usa pesos alpha por posicao espacial para capturar melhor
    multiplas regioes ativas — mais adequado para patologias bilaterais.
    """
    with tf.GradientTape() as tape:
        batch = tf.expand_dims(img_tensor,0)
        conv_out,preds = grad_model(batch)
        loss = preds[:,class_idx]
    grads    = tape.gradient(loss,conv_out)[0]   # (H,W,C)
    acts     = conv_out[0]                        # (H,W,C)
    grad2    = grads**2
    grad3    = grads**3
    A_sum    = tf.reduce_sum(acts,axis=(0,1),keepdims=True)  # (1,1,C)
    alpha    = grad2/(2*grad2 + A_sum*grad3 + 1e-7)
    weights  = tf.reduce_sum(alpha*tf.nn.relu(grads),axis=(0,1))  # (C,)
    hm       = acts @ weights[...,tf.newaxis]    # (H,W,1)
    hm       = tf.squeeze(hm)
    hm       = tf.maximum(hm,0)/(tf.math.reduce_max(hm)+1e-7)
    return hm.numpy(), float(preds[0,class_idx])


def heatmap_to_bbox(heatmap, threshold=0.4, pad=10):
    """Bounding box a partir do heatmap para prompt MedSAM."""
    hm_up = np.array(
        PILImage.fromarray((heatmap*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE))
    )/255.0
    ys,xs = np.where(hm_up>=threshold)
    if len(xs)==0: return [0,0,IMG_SIZE,IMG_SIZE]
    return [max(int(xs.min())-pad,0), max(int(ys.min())-pad,0),
            min(int(xs.max())+pad,IMG_SIZE), min(int(ys.max())+pad,IMG_SIZE)]


def overlay_heatmap(pil_img, heatmap, alpha=0.45):
    """Sobrepoe heatmap colorido sobre imagem grayscale."""
    orig = np.array(pil_img.convert('L').resize((IMG_SIZE,IMG_SIZE)))
    orig_rgb = np.stack([orig]*3,axis=-1)
    hm_up = np.array(PILImage.fromarray((heatmap*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE)))
    hm_col = (plt.cm.jet(hm_up/255.0)[:,:,:3]*255).astype(np.uint8)
    return PILImage.fromarray((orig_rgb*(1-alpha)+hm_col*alpha).astype(np.uint8))


LABEL_SEG_COLORS = {
    'atelectasis':     (46,134,171),
    'cardiomegaly':    (231,111,81),
    'pleural_effusion':(40,167,69),
    'pneumothorax':    (255,193,7),
    'consolidation':   (108,99,255),
    'edema':           (220,53,69),
}

def run_medsam(pil_img, bbox, label='pneumothorax'):
    """
    Segmentacao real pixel-a-pixel via MedSAM.
    Usa bbox como prompt. Fallback visual se checkpoint nao carregado.
    """
    img_gray = np.array(pil_img.convert('L').resize((IMG_SIZE,IMG_SIZE)))
    img_rgb  = np.stack([img_gray]*3,axis=-1).astype(np.uint8)
    color    = LABEL_SEG_COLORS.get(label,(255,193,7))

    if MEDSAM_READY and bbox:
        try:
            medsam_predictor.set_image(img_rgb)
            masks,_,_ = medsam_predictor.predict(
                point_coords=None, point_labels=None,
                box=np.array(bbox,dtype=float)[None,:],
                multimask_output=False)
            mask    = masks[0]
            overlay = img_rgb.copy().astype(np.float32)
            for c,v in enumerate(color):
                overlay[:,:,c] = np.where(mask,
                    overlay[:,:,c]*0.55+v*0.45,
                    overlay[:,:,c])
            # Contorno da mascara
            from PIL import ImageFilter
            mp   = PILImage.fromarray((mask*255).astype(np.uint8))
            cont = np.array(mp.filter(ImageFilter.FIND_EDGES))>0
            for c in range(3): overlay[:,:,c] = np.where(cont,color[c],overlay[:,:,c])
            return PILImage.fromarray(overlay.clip(0,255).astype(np.uint8))
        except Exception as e:
            print(f'MedSAM error: {e}')

    # Fallback
    from PIL import ImageDraw
    pil  = PILImage.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil,'RGBA')
    if bbox:
        x0,y0,x1,y1=bbox
        draw.rectangle([x0,y0,x1,y1],fill=(*color,50),outline=(*color,220),width=3)
    draw.text((8,8),'MedSAM (sem checkpoint)',fill=(255,80,80,200))
    return pil


print('Funcoes definidas: Grad-CAM | Grad-CAM++ | MedSAM (segmentacao real)')

Funcoes definidas: Grad-CAM | Grad-CAM++ | MedSAM (segmentacao real)


## SEÇÃO 5 — Função principal do Gradio

In [ ]:
from PIL import Image as PILImage
import numpy as np

def preprocess(pil_img):
    img = np.array(pil_img.convert('L').resize((IMG_SIZE,IMG_SIZE)),dtype=np.float32)/255.0
    img = (img-MEAN)/STD
    img = np.stack([img]*3, axis=-1)
    return tf.constant(img, dtype=tf.float32)


def get_gradcam(img_tensor, class_idx):
    """Grad-CAM padrão."""
    with tf.GradientTape() as tape:
        batch = tf.expand_dims(img_tensor,0)
        conv_out,preds = grad_model(batch)
        loss = preds[:,class_idx]
    grads  = tape.gradient(loss,conv_out)
    pooled = tf.reduce_mean(grads,axis=(0,1,2))
    hm     = conv_out[0] @ pooled[...,tf.newaxis]
    hm     = tf.squeeze(hm)
    hm     = tf.maximum(hm,0)/(tf.math.reduce_max(hm)+1e-7)
    return hm.numpy(), float(preds[0,class_idx])


def get_gradcam_pp(img_tensor, class_idx):
    """
    Grad-CAM++ (Chattopadhay et al., 2018).
    Usa pesos alpha por posicao espacial para capturar melhor
    multiplas regioes ativas — mais adequado para patologias bilaterais.
    """
    with tf.GradientTape() as tape:
        batch = tf.expand_dims(img_tensor,0)
        conv_out,preds = grad_model(batch)
        loss = preds[:,class_idx]
    grads    = tape.gradient(loss,conv_out)[0]   # (H,W,C)
    acts     = conv_out[0]                        # (H,W,C)
    grad2    = grads**2
    grad3    = grads**3
    A_sum    = tf.reduce_sum(acts,axis=(0,1),keepdims=True)  # (1,1,C)
    alpha    = grad2/(2*grad2 + A_sum*grad3 + 1e-7)
    weights  = tf.reduce_sum(alpha*tf.nn.relu(grads),axis=(0,1))  # (C,)
    hm       = acts @ weights[...,tf.newaxis]    # (H,W,1)
    hm       = tf.squeeze(hm)
    hm       = tf.maximum(hm,0)/(tf.math.reduce_max(hm)+1e-7)
    return hm.numpy(), float(preds[0,class_idx])


def heatmap_to_bbox(heatmap, threshold=0.4, pad=10):
    """Bounding box a partir do heatmap para prompt MedSAM."""
    hm_up = np.array(
        PILImage.fromarray((heatmap*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE))
    )/255.0
    ys,xs = np.where(hm_up>=threshold)
    if len(xs)==0: return [0,0,IMG_SIZE,IMG_SIZE]
    return [max(int(xs.min())-pad,0), max(int(ys.min())-pad,0),
            min(int(xs.max())+pad,IMG_SIZE), min(int(ys.max())+pad,IMG_SIZE)]


def overlay_heatmap(pil_img, heatmap, alpha=0.45):
    """Sobrepoe heatmap colorido sobre imagem grayscale."""
    orig = np.array(pil_img.convert('L').resize((IMG_SIZE,IMG_SIZE)))
    orig_rgb = np.stack([orig]*3,axis=-1)
    hm_up = np.array(PILImage.fromarray((heatmap*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE)))
    hm_col = (plt.cm.jet(hm_up/255.0)[:,:,:3]*255).astype(np.uint8)
    return PILImage.fromarray((orig_rgb*(1-alpha)+hm_col*alpha).astype(np.uint8))


LABEL_SEG_COLORS = {
    'atelectasis':     (46,134,171),
    'cardiomegaly':    (231,111,81),
    'pleural_effusion':(40,167,69),
    'pneumothorax':    (255,193,7),
    'consolidation':   (108,99,255),
    'edema':           (220,53,69),
}

def run_medsam(pil_img, bbox, label='pneumothorax'):
    """
    Segmentacao real pixel-a-pixel via MedSAM.
    Usa bbox como prompt. Fallback visual se checkpoint nao carregado.
    """
    img_gray = np.array(pil_img.convert('L').resize((IMG_SIZE,IMG_SIZE)))
    img_rgb  = np.stack([img_gray]*3,axis=-1).astype(np.uint8)
    color    = LABEL_SEG_COLORS.get(label,(255,193,7))

    if MEDSAM_READY and bbox:
        try:
            medsam_predictor.set_image(img_rgb)
            masks,_,_ = medsam_predictor.predict(
                point_coords=None, point_labels=None,
                box=np.array(bbox,dtype=float)[None,:],
                multimask_output=False)
            mask    = masks[0]
            overlay = img_rgb.copy().astype(np.float32)
            for c,v in enumerate(color):
                overlay[:,:,c] = np.where(mask,
                    overlay[:,:,c]*0.55+v*0.45,
                    overlay[:,:,c])
            # Contorno da mascara
            from PIL import ImageFilter
            mp   = PILImage.fromarray((mask*255).astype(np.uint8))
            cont = np.array(mp.filter(ImageFilter.FIND_EDGES))>0
            for c in range(3): overlay[:,:,c] = np.where(cont,color[c],overlay[:,:,c])
            return PILImage.fromarray(overlay.clip(0,255).astype(np.uint8))
        except Exception as e:
            print(f'MedSAM error: {e}')

    # Fallback
    from PIL import ImageDraw
    pil  = PILImage.fromarray(img_rgb)
    draw = ImageDraw.Draw(pil,'RGBA')
    if bbox:
        x0,y0,x1,y1=bbox
        draw.rectangle([x0,y0,x1,y1],fill=(*color,50),outline=(*color,220),width=3)
    draw.text((8,8),'MedSAM (sem checkpoint)',fill=(255,80,80,200))
    return pil


print('Funcoes definidas: Grad-CAM | Grad-CAM++ | MedSAM (segmentacao real)')

def analyze_cxr(image, threshold):
    """
    Funcao principal do Gradio.
    Retorna: original | Grad-CAM | Grad-CAM++ | MedSAM | tabela HTML
    """
    if image is None:
        return None, None, None, None, 'Nenhuma imagem enviada.'

    img_tensor = preprocess(image)
    probs   = model_m2(tf.expand_dims(img_tensor,0), training=False).numpy()[0]
    top_idx = int(np.argmax(probs))
    top_lbl = LABELS[top_idx]

    predictions = [{'label':lbl,'label_pt':LABEL_PT[lbl],
                    'prob':float(probs[i]),'positive':float(probs[i])>=threshold}
                   for i,lbl in enumerate(LABELS)]

    # Grad-CAM e Grad-CAM++
    hm,    conf    = get_gradcam(img_tensor, top_idx)
    hm_pp, conf_pp = get_gradcam_pp(img_tensor, top_idx)

    # bbox do Grad-CAM++ como prompt MedSAM
    bbox_pp = heatmap_to_bbox(hm_pp)

    # Imagens
    orig_rgb    = PILImage.fromarray(
        np.stack([np.array(image.convert('L').resize((IMG_SIZE,IMG_SIZE)))]*3, axis=-1))
    img_gradcam    = overlay_heatmap(image, hm)
    img_gradcam_pp = overlay_heatmap(image, hm_pp)
    img_medsam     = run_medsam(image, bbox_pp, label=top_lbl)

    # Tabela HTML
    rows = ''
    for p in sorted(predictions, key=lambda x: -x['prob']):
        pct   = p['prob']*100
        color = LABEL_COLORS_HEX.get(p['label'],'#888')
        bg    = '#1e3a2b' if p['positive'] else '#1e1e2e'
        badge = ('<span style="color:#4ade80;font-weight:bold">POSITIVO</span>'
                 if p['positive'] else
                 '<span style="color:#6b7280">negativo</span>')
        bar   = (f'<div style="background:#334155;border-radius:4px;height:8px;width:100%">'
                 f'<div style="background:{color};width:{pct:.1f}%;height:8px;'
                 f'border-radius:4px"></div></div>')
        rows += (f'<tr style="background:{bg}">'
                 f'<td style="padding:8px 12px;color:#e2e8f0">{p["label_pt"]}</td>'
                 f'<td style="padding:8px 12px;text-align:center;font-family:monospace;'
                 f'color:#93c5fd">{pct:.1f}%</td>'
                 f'<td style="padding:8px 12px;width:140px">{bar}</td>'
                 f'<td style="padding:8px 12px;text-align:center">{badge}</td>'
                 f'</tr>')

    positivos = [p for p in predictions if p['positive']]
    summary = (f"<b style='color:#4ade80'>{len(positivos)} achado(s):</b> "
               + ", ".join(f"{p['label_pt']} ({p['prob']:.0%})"
                            for p in sorted(positivos, key=lambda x: -x['prob']))
               if positivos else
               "<span style='color:#9ca3af'>Nenhum achado positivo.</span>")

    sam_badge = 'MedSAM ativo' if MEDSAM_READY else 'MedSAM fallback (bbox)'
    html = f"""
    <div style='background:#0f172a;border-radius:12px;padding:16px;font-family:sans-serif'>
      <div style='color:#94a3b8;font-size:12px;margin-bottom:12px;
                  text-transform:uppercase;letter-spacing:1px'>
        Limiar: {threshold:.0%} | {sam_badge}
      </div>
      <table style='width:100%;border-collapse:collapse'>
        <thead><tr style='background:#1e293b'>
          <th style='padding:8px 12px;text-align:left;color:#64748b;font-size:11px'>PATOLOGIA</th>
          <th style='padding:8px 12px;color:#64748b;font-size:11px'>PROB</th>
          <th style='padding:8px 12px;color:#64748b;font-size:11px'></th>
          <th style='padding:8px 12px;color:#64748b;font-size:11px'>STATUS</th>
        </tr></thead>
        <tbody>{rows}</tbody>
      </table>
      <div style='margin-top:12px;padding:10px;background:#1e293b;border-radius:8px;
                  color:#e2e8f0;font-size:13px'>{summary}</div>
      <div style='margin-top:8px;color:#475569;font-size:11px'>
        Grad-CAM: <b style='color:#93c5fd'>{LABEL_PT[top_lbl]}</b> (p={conf:.1%}) |
        Grad-CAM++ (p={conf_pp:.1%}) | prompt MedSAM: bbox GradCAM++
      </div>
    </div>
    """
    return orig_rgb, img_gradcam, img_gradcam_pp, img_medsam, html


print('analyze_cxr definida.')

Funcoes definidas: Grad-CAM | Grad-CAM++ | MedSAM (segmentacao real)
analyze_cxr definida.


## SEÇÃO 6 — Interface Gradio

In [ ]:
import shutil

# Copia exemplos para /tmp para evitar erro de I/O do Drive
examples = []
try:
    import pandas as pd
    df = pd.read_csv(f'{BASE}/datasets_pi_2026/estruturacao/Stanford_Cropping/chex_crop_valid.csv')
    os.makedirs('/tmp/cxr_examples', exist_ok=True)
    for i in range(min(4, len(df))):
        src = df['img_path'].iloc[i]
        dst = f'/tmp/cxr_examples/example_{i}.png'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
        if os.path.exists(dst):
            examples.append([dst])
    print(f'{len(examples)} exemplos copiados para /tmp.')
except Exception as e:
    print(f'Exemplos nao disponiveis: {e}')

css = """
.gradio-container { background: #0f172a !important; }
.main-header {
  background: linear-gradient(135deg,#1e3a5f,#0f172a);
  border-radius:12px; padding:20px 24px; margin-bottom:16px;
  border:1px solid #1e40af40;
}
footer { display:none !important; }
"""

with gr.Blocks(theme=gr.themes.Base(), css=css, title='CXR Analyzer') as demo:

    gr.HTML("""
    <div class='main-header'>
      <h1 style='margin:0;color:#e2e8f0;font-size:22px;font-weight:700'>CXR Analyzer</h1>
      <p style='margin:4px 0 0;color:#64748b;font-size:13px'>
        Xception M2 U-Ignore | Grad-CAM | Grad-CAM++ | MedSAM
      </p>
    </div>
    """)

    with gr.Row():
        # Coluna esquerda — input
        with gr.Column(scale=1):
            img_input = gr.Image(
                label='Raio-X de Torax',
                type='pil',
                sources=['upload','clipboard'],
                height=280,
            )
            threshold_slider = gr.Slider(
                minimum=0.1, maximum=0.9, value=0.5, step=0.05,
                label='Limiar de Classificacao',
                info='Probabilidade minima para classificar como positivo'
            )
            btn = gr.Button('Analisar Raio-X', variant='primary', size='lg')

            if examples:
                gr.Examples(examples=examples, inputs=[img_input],
                            label='Exemplos do Valid Set CheXpert')

        # Coluna direita — resultados
        with gr.Column(scale=2):
            results_html = gr.HTML(label='Resultados')

            with gr.Row():
                img_orig     = gr.Image(label='Original',      height=240, show_download_button=True)
                img_gradcam  = gr.Image(label='Grad-CAM',      height=240, show_download_button=True)
                img_gradcampp= gr.Image(label='Grad-CAM++',    height=240, show_download_button=True)
                img_medsam   = gr.Image(label='MedSAM (seg)',  height=240, show_download_button=True)

    outputs = [img_orig, img_gradcam, img_gradcampp, img_medsam, results_html]

    btn.click(fn=analyze_cxr, inputs=[img_input, threshold_slider], outputs=outputs)
    threshold_slider.change(fn=analyze_cxr, inputs=[img_input, threshold_slider], outputs=outputs)

print('Interface Gradio montada.')
print('4 paineis: Original | Grad-CAM | Grad-CAM++ | MedSAM')
print('Rodando na proxima celula...')

2 exemplos copiados para /tmp.
Interface Gradio montada.
4 paineis: Original | Grad-CAM | Grad-CAM++ | MedSAM
Rodando na proxima celula...


## SEÇÃO 7 — Lança o App

In [ ]:
# share=True gera link público temporário (~72h)
demo.launch(share=True, debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3735f9e32816232689.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


KeyboardInterrupt: 

In [ ]:
# Rodar esta célula para encerrar o servidor e liberar o link
demo.close()
print('Servidor encerrado.')

Closing server running on port: 7860
Servidor encerrado.
